# Notebook 3: Qwen3-VL-8B-Instruct QLoRA — Single-Stage SFT on VizWiz-LF

**Architecture:** Qwen3-VL-8B-Instruct with 4-bit NF4 QLoRA on `q_proj` and `v_proj`  
**Strategy:** Single-Stage Supervised Fine-Tuning on LF.json synthetic answers  
**Environment:** Kaggle Dual-T4 (2×16 GB VRAM)  

### Key Fixes Applied
- `image.thumbnail((512,512))` to prevent OOM from dynamic-resolution patches
- `mm_token_type_ids` extracted in `__getitem__` to prevent M-RoPE crash
- Custom collator uses `torch.cat` for `pixel_values`/`image_grid_thw`, `torch.stack` for sequence tensors
- `KaggleDiskSaverCallback` deletes previous checkpoint before saving new one
- `HubPushCallback` pushes checkpoints + metrics every epoch
- Modern AMP syntax: `torch.amp.GradScaler('cuda')` / `torch.amp.autocast('cuda')`
- `warmup_steps=50` (no deprecated `warmup_ratio`)

## Cell 1 — Install Dependencies

In [1]:
# ============================================================
# CELL 1 — Install Dependencies
# NOTE: Pillow is intentionally EXCLUDED from the upgrade list.
#       Upgrading Kaggle's native Pillow breaks torchvision
#       with: ImportError: cannot import name '_Ink'
# ============================================================
!pip install -q --upgrade \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    datasets \
    huggingface_hub \
    evaluate \
    bert-score \
    rouge-score \
    nltk \
    sacrebleu \
    opencv-python-headless

# Download NLTK data needed for METEOR
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
print("✅ Dependencies installed.")

✅ Dependencies installed.


## Cell 2 — Imports & Global Config

In [ ]:
# ============================================================
# CELL 2 — Imports & Global Configuration
# ============================================================
import os, json, shutil, time, gc, csv
from pathlib import Path
from typing import Any, Dict, List, Optional

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,   # NOT AutoModelForVision2Seq (deprecated)
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerState,
    TrainerControl,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login

# ── Paths ────────────────────────────────────────────────────
DATA_ROOT   = Path("/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf")          # adjust if dataset name differs
LF_JSON     = "/kaggle/input/datasets/f230017abdullahkhan/viz-wiz-standard-with-lf/LF.json"   # Only ONE file exists
IMG_ROOT    = DATA_ROOT           # root containing train/train/ and val/val/
OUTPUT_DIR  = Path("/kaggle/working/qwen3vl_qlora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERT_HOLDOUT_PATH = OUTPUT_DIR / "expert_holdout.json"

# ── Model ────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-VL-8B-Instruct"
HUB_REPO    = "Abdullah-Khan-Niazi/vizwiz-lf-qwen3vl"  # ← EDIT THIS
HF_TOKEN    = ''         # set in Kaggle Secrets

# ── Training Hyperparameters ─────────────────────────────────
BATCH_SIZE   = 1      # MUST be 1 for Qwen3-VL dynamic resolution
GRAD_ACCUM   = 16     # effective batch = 16
MAX_SEQ_LEN  = 2048
LR           = 2e-4
NUM_EPOCHS   = 3
WARMUP_STEPS = 50
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42

torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True  # speed boost
print(f"✅ Config ready. CUDA devices: {torch.cuda.device_count()}")

✅ Config ready. CUDA devices: 2


## Cell 3 — High-Speed Image Path Indexer

In [3]:
# ============================================================
# CELL 3 — High-Speed Image Path Indexer (RAM Dictionary)
#
# CRITICAL: Build IMAGE_PATH_MAP once at startup.
# NEVER use os.path.exists() or glob inside __getitem__.
# That causes CPU starvation and massive latency.
# ============================================================
IMAGE_PATH_MAP: Dict[str, str] = {}

for split_dir in ["train/train", "val/val"]:
    split_path = IMG_ROOT / split_dir
    if split_path.exists():
        for img_file in split_path.iterdir():
            if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".gif", ".bmp"):
                # filename → absolute path string (no duplicates; first-found wins)
                if img_file.name not in IMAGE_PATH_MAP:
                    IMAGE_PATH_MAP[img_file.name] = str(img_file)

print(f"✅ IMAGE_PATH_MAP built: {len(IMAGE_PATH_MAP):,} images indexed.")

✅ IMAGE_PATH_MAP built: 31,704 images indexed.


## Cell 4 — Parse LF.json & Create Train/Holdout Split

In [4]:
# ============================================================
# CELL 4 — Parse LF.json & Create Train/Holdout Split
#
# RULES:
#   • There is NO LF_train.json / LF_val.json — only LF.json.
#   • Sources containing 'expert' or 'human'  → expert_holdout
#   • All other sources (GPT-4V, LLaVA, …)   → synthetic_train
#   • Models train ONLY on synthetic_train.
# ============================================================
import os
import json

with open(LF_JSON, "r", encoding="utf-8") as f:
    lf_data = json.load(f)

synthetic_train: List[Dict] = []
expert_holdout:  List[Dict] = []

# Safely handle if LF.json is a list OR a dictionary
items = lf_data if isinstance(lf_data, list) else list(lf_data.values())

for item in items:
    # Extract filename from "image_url" (fallback to "image" just in case)
    img_url = item.get("image_url", item.get("image", ""))
    image_name = os.path.basename(img_url) if img_url else ""
    
    question = item.get("question", "").strip()
    long_answers = item.get("long_answers", {})

    if not image_name or not question or not long_answers:
        continue

    # Skip if image not found in our high-speed RAM index
    if image_name not in IMAGE_PATH_MAP:
        continue

    # Process the nested long_answers dictionary
    for source, answer_data in long_answers.items():
        if isinstance(answer_data, dict):
            answer_text = answer_data.get("answer_paragraph", "")
        else:
            answer_text = str(answer_data)
            
        if not answer_text or not answer_text.strip():
            continue

        record = {
            "image":    image_name,
            "question": question,
            "answer":   answer_text.strip(),
            "source":   source,
        }

        src_lower = source.lower()
        if "expert" in src_lower or "human" in src_lower:
            expert_holdout.append(record)
        else:
            synthetic_train.append(record)

# Persist expert holdout immediately so it survives any crash
with open(EXPERT_HOLDOUT_PATH, "w", encoding="utf-8") as f:
    json.dump(expert_holdout, f, indent=2, ensure_ascii=False)

print(f"✅ synthetic_train : {len(synthetic_train):,} samples")
print(f"✅ expert_holdout  : {len(expert_holdout):,} samples  (saved → {EXPERT_HOLDOUT_PATH})")

✅ synthetic_train : 3,596 samples
✅ expert_holdout  : 600 samples  (saved → /kaggle/working/qwen3vl_qlora/expert_holdout.json)


## Cell 5 — Load Processor & Model with 4-bit QLoRA

In [5]:
# ============================================================
# CELL 5 — Load Processor & Qwen3-VL-8B in 4-bit NF4 QLoRA
# ============================================================

# Log in to HuggingFace Hub
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ HF login successful.")
else:
    print("⚠️  HF_TOKEN not set — Hub push callbacks will be skipped.")

# ── Processor ────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"

# ── 4-bit NF4 Quantisation Config ────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Load Model ───────────────────────────────────────────────
# Use AutoModelForImageTextToText (NOT deprecated AutoModelForVision2Seq)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Prepare model for k-bit training (required before PEFT wrapping)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# ── LoRA Configuration ────────────────────────────────────────
# Target ONLY q_proj and v_proj as specified
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Qwen3-VL-8B loaded with 4-bit NF4 QLoRA.")

✅ HF login successful.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 7,667,712 || all params: 8,774,791,408 || trainable%: 0.0874
✅ Qwen3-VL-8B loaded with 4-bit NF4 QLoRA.


In [6]:
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# Create a folder in Kaggle's fast working directory
PRECOMPUTED_DIR = "/kaggle/working/qwen_precomputed"
os.makedirs(PRECOMPUTED_DIR, exist_ok=True)

print("Pre-computing all Qwen-VL inputs to bypass CPU bottlenecks...")

def precompute_records(records, prefix):
    for idx, rec in enumerate(tqdm(records, desc=f"Processing {prefix}")):
        save_path = os.path.join(PRECOMPUTED_DIR, f"{prefix}_{idx}.pt")
        
        # Skip if already processed (in case of a notebook restart)
        if os.path.exists(save_path):
            continue
            
        fname = rec["image"]
        img_path = IMAGE_PATH_MAP.get(fname)
        
        # 1. Load Image
        if img_path is None:
            pil_image = Image.fromarray(np.full((224, 224, 3), 128, dtype=np.uint8))
        else:
            try:
                pil_image = Image.open(img_path).convert("RGB")
                pil_image.thumbnail((512, 512))
            except:
                pil_image = Image.fromarray(np.full((224, 224, 3), 128, dtype=np.uint8))

        # 2. Build Chat
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": rec["question"]}]},
            {"role": "assistant", "content": [{"type": "text", "text": rec["answer"]}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        # 3. Process
        inputs = processor(
            text=[text],
            images=[pil_image],
            return_tensors="pt",
            max_length=MAX_SEQ_LEN,
            truncation=True,
            padding=False,
        )

        # 4. Squeeze batch dims and compress pixel_values to FP16
        processed = {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "pixel_values": inputs["pixel_values"].to(torch.float16), 
            "image_grid_thw": inputs["image_grid_thw"],
        }
        
        if "mm_token_type_ids" in inputs:
            processed["mm_token_type_ids"] = inputs["mm_token_type_ids"].squeeze(0)
        else:
            processed["mm_token_type_ids"] = torch.zeros_like(processed["input_ids"])

        # Save as a fast PyTorch file
        torch.save(processed, save_path)

# Run this on your synthetic_train list
precompute_records(synthetic_train, "train")

Pre-computing all Qwen-VL inputs to bypass CPU bottlenecks...


Processing train:   0%|          | 0/3596 [00:00<?, ?it/s]

## Cell 6 — Dataset Class with Thumbnail Downscale & mm_token_type_ids

In [7]:
# ============================================================
# CELL 6 — VizWizLFDataset (ULTRA-FAST PRECOMPUTED VERSION)
# ============================================================

IGNORE_INDEX = -100  

class VizWizLFDataset(Dataset):
    def __init__(
        self,
        records: List[Dict],
        processor: AutoProcessor,
        prefix: str = "train",
    ):
        self.records = records
        self.processor = processor
        self.prefix = prefix
        
        # Cache the assistant token sequence once in the init
        self.asst_ids = self.processor.tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
        self.asst_len = len(self.asst_ids)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # ── 1. Lightning-fast load from disk ─────────────────────────────
        load_path = os.path.join("/kaggle/working/qwen_precomputed", f"{self.prefix}_{idx}.pt")
        data = torch.load(load_path, weights_only=True)
        
        input_ids         = data["input_ids"]
        attention_mask    = data["attention_mask"]
        image_grid_thw    = data["image_grid_thw"]
        mm_token_type_ids = data["mm_token_type_ids"]
        
        # Cast pixels back to FP32 for the model to process
        pixel_values = data["pixel_values"].to(torch.float32)

        # ── 2. Create labels — mask the user turn ─────────────────────────
        labels = input_ids.clone()

        asst_start = -1
        ids_list = input_ids.tolist()
        
        # Scan for the assistant header
        for i in range(len(ids_list) - self.asst_len + 1):
            if ids_list[i : i + self.asst_len] == self.asst_ids:
                asst_start = i + self.asst_len  # answer begins right after the header
                break

        if asst_start == -1:
            labels[:] = IGNORE_INDEX
        else:
            labels[:asst_start] = IGNORE_INDEX  # mask prompt tokens

        return {
            "input_ids":          input_ids,
            "attention_mask":     attention_mask,
            "labels":             labels,
            "pixel_values":       pixel_values,
            "image_grid_thw":     image_grid_thw,
            "mm_token_type_ids":  mm_token_type_ids,
        }

print("✅ Ultra-fast VizWizLFDataset class defined.")

✅ Ultra-fast VizWizLFDataset class defined.


## Cell 7 — Custom Data Collator

In [8]:
# ============================================================
# CELL 7 — Custom Qwen3-VL Data Collator
#
# CRITICAL COLLATION RULES:
#   • pixel_values    → torch.cat(dim=0)   — variable # of patches per image
#   • image_grid_thw  → torch.cat(dim=0)   — shape metadata, one row per image
#   • input_ids       → torch.stack(dim=0) — fixed seq_len after truncation
#   • attention_mask  → torch.stack(dim=0)
#   • labels          → torch.stack(dim=0)
#   • mm_token_type_ids → torch.stack(dim=0)
#
# Using stack where cat is needed (or vice versa) causes immediate
# shape errors inside the model's attention/RoPE layers.
# ============================================================

def custom_qwen_data_collator(features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """
    Custom collator for Qwen3-VL dynamic-resolution inputs.

    features: list of dicts returned by VizWizLFDataset.__getitem__
    """
    # Sequence tensors — all same length (truncated to MAX_SEQ_LEN), so stack
    input_ids         = torch.stack([f["input_ids"]         for f in features], dim=0)
    attention_mask    = torch.stack([f["attention_mask"]    for f in features], dim=0)
    labels            = torch.stack([f["labels"]            for f in features], dim=0)
    mm_token_type_ids = torch.stack([f["mm_token_type_ids"] for f in features], dim=0)

    # Vision tensors — variable number of patches per image, so cat along dim=0
    pixel_values  = torch.cat([f["pixel_values"]  for f in features], dim=0)
    image_grid_thw = torch.cat([f["image_grid_thw"] for f in features], dim=0)

    return {
        "input_ids":          input_ids,
        "attention_mask":     attention_mask,
        "labels":             labels,
        "pixel_values":       pixel_values,
        "image_grid_thw":     image_grid_thw,
        "mm_token_type_ids":  mm_token_type_ids,
    }

print("✅ custom_qwen_data_collator defined.")

✅ custom_qwen_data_collator defined.


## Cell 8 — Callbacks (KaggleDiskSaver + HubPush)

In [9]:
# ============================================================
# CELL 8 — Trainer Callbacks
#
# KaggleDiskSaverCallback:
#   Kaggle has only ~19.5 GB free disk space. If we keep every
#   checkpoint, we overflow and the job crashes. This callback
#   deletes the PREVIOUS checkpoint dir before the Trainer
#   writes the new one.
#
# HubPushCallback:
#   After every epoch: push model weights, loss CSV, and a
#   metrics JSON to the HuggingFace Hub repo.
# ============================================================

class KaggleDiskSaverCallback(TrainerCallback):
    """
    Prevents Kaggle disk overflow by deleting the previous
    checkpoint directory before saving the new one.
    """
    def __init__(self, output_dir: str):
        self.output_dir  = Path(output_dir)
        self._last_ckpt: Optional[Path] = None

    def on_save(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        # Find all existing checkpoint-* directories
        ckpt_dirs = sorted(
            self.output_dir.glob("checkpoint-*"),
            key=lambda p: int(p.name.split("-")[-1]),
        )
        # Delete all but the most recently saved one
        for old_ckpt in ckpt_dirs[:-1]:
            print(f"  🗑️  KaggleDiskSaver: removing {old_ckpt}")
            shutil.rmtree(old_ckpt, ignore_errors=True)


class HubPushCallback(TrainerCallback):
    """
    Pushes model checkpoint + metrics to HuggingFace Hub
    at the end of every epoch.
    """
    def __init__(self, hub_repo: str, output_dir: str, token: str):
        self.hub_repo   = hub_repo
        self.output_dir = Path(output_dir)
        self.token      = token
        self.api        = HfApi() if token else None
        self.loss_log: List[Dict] = []

    def on_log(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs: Optional[Dict] = None,
        **kwargs,
    ):
        if logs:
            self.loss_log.append({"step": state.global_step, **logs})

    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        if not self.api or not self.token:
            print("  ⚠️  HubPushCallback: no HF token, skipping push.")
            return

        epoch = int(state.epoch)
        print(f"  📤 HubPushCallback: pushing epoch {epoch} to {self.hub_repo} …")

        # Save loss CSV
        loss_csv = self.output_dir / "loss_log.csv"
        with open(loss_csv, "w", newline="") as f:
            if self.loss_log:
                writer = csv.DictWriter(f, fieldnames=self.loss_log[0].keys())
                writer.writeheader()
                writer.writerows(self.loss_log)

        # Push loss CSV
        self.api.upload_file(
            path_or_fileobj=str(loss_csv),
            path_in_repo="loss_log.csv",
            repo_id=self.hub_repo,
            repo_type="model",
            token=self.token,
        )

        # Push latest checkpoint folder if it exists
        ckpt_dirs = sorted(
            self.output_dir.glob("checkpoint-*"),
            key=lambda p: int(p.name.split("-")[-1]),
        )
        if ckpt_dirs:
            latest = ckpt_dirs[-1]
            self.api.upload_folder(
                folder_path=str(latest),
                path_in_repo=f"epoch_{epoch}",
                repo_id=self.hub_repo,
                repo_type="model",
                token=self.token,
            )
        print(f"  ✅ Hub push complete for epoch {epoch}.")


print("✅ KaggleDiskSaverCallback and HubPushCallback defined.")

✅ KaggleDiskSaverCallback and HubPushCallback defined.


## Cell 9 — Instantiate Dataset & DataLoader

In [10]:
# ============================================================
# CELL 9 — Instantiate Datasets and DataLoaders
#
# DataLoader settings:
#   pin_memory=True        — avoids extra H2D copies
#   prefetch_factor=2      — keeps GPU fed
#   persistent_workers=True — avoids worker respawn overhead
# ============================================================

train_dataset = VizWizLFDataset(synthetic_train, processor, prefix="train")
# Quick sanity check on one sample
sample = train_dataset[0]
print("Sample keys:", list(sample.keys()))
for k, v in sample.items():
    print(f"  {k}: {v.shape}  dtype={v.dtype}")

print(f"\n✅ train_dataset: {len(train_dataset):,} samples")

Sample keys: ['input_ids', 'attention_mask', 'labels', 'pixel_values', 'image_grid_thw', 'mm_token_type_ids']
  input_ids: torch.Size([389])  dtype=torch.int64
  attention_mask: torch.Size([389])  dtype=torch.int64
  labels: torch.Size([389])  dtype=torch.int64
  pixel_values: torch.Size([768, 1536])  dtype=torch.float32
  image_grid_thw: torch.Size([1, 3])  dtype=torch.int64
  mm_token_type_ids: torch.Size([389])  dtype=torch.int64

✅ train_dataset: 3,596 samples


## Cell 10 — Training Arguments & Trainer

In [11]:
# ============================================================
# CELL 10 — TrainingArguments & HF Trainer
#
# Deprecation fixes:
#   • warmup_steps=50  (NOT warmup_ratio — deprecated in HF v5.2)
#   • fp16=False / bf16=True (T4 supports BF16 via bitsandbytes)
# ============================================================

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,          # NOT warmup_ratio
    lr_scheduler_type="cosine",
    bf16=True,                           # T4 supports BF16 via bitsandbytes
    fp16=False,
    optim="paged_adamw_8bit",           # memory-efficient optimiser
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,               # Saves your progress every ~50 steps
    save_total_limit=1,          # Keeps the last 2 checkpoints safely on disk
    load_best_model_at_end=False,
    remove_unused_columns=False,         # MUST be False for custom collator
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    dataloader_persistent_workers=True,
    report_to="none",                    # disable W&B etc.
    seed=SEED,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# ── Callbacks ────────────────────────────────────────────────
callbacks = [
    KaggleDiskSaverCallback(output_dir=str(OUTPUT_DIR)),
    HubPushCallback(
        hub_repo=HUB_REPO,
        output_dir=str(OUTPUT_DIR),
        token=HF_TOKEN,
    ),
]

# ── Trainer ──────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=custom_qwen_data_collator,
    callbacks=callbacks,
)

print("✅ Trainer configured.")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configured.


## Cell 11 — Train

In [12]:
import gc
import torch

# Delete model/optimizer if they exist in the current namespace
if 'model' in locals(): del model
if 'optimizer' in locals(): del optimizer

gc.collect()
torch.cuda.empty_cache()
# Set this flag to help with memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("VRAM Purged.")

VRAM Purged.


In [ ]:
import os
import glob
import time

# ── 1. Auto-Resume Logic ──────────────────────────────────────────────────
def find_latest_checkpoint(base_dir):
    # Search for directories named 'checkpoint-XXXX'
    checkpoint_dirs = glob.glob(os.path.join(base_dir, "checkpoint-*"))
    if not checkpoint_dirs:
        return None
    
    # Return the one with the highest step number
    return max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))

latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)

# ── 2. Run Training ────────────────────────────────────────────────────────
print("🚀 Starting Single-Stage SFT on VizWiz-LF synthetic data …")

if latest_checkpoint:
    print(f"🔄 Auto-Resume: Found checkpoint at {latest_checkpoint}")
    print("📥 Resuming training from last saved state...")
else:
    print("🆕 No checkpoint found. Starting fresh training...")

t0 = time.time()

# Pass the path to resume_from_checkpoint
# If latest_checkpoint is None, it starts from scratch automatically
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

elapsed = time.time() - t0
print(f"\n✅ Training complete in {elapsed/3600:.2f} h")
print(f"   train_loss: {train_result.training_loss:.4f}")

# ── 3. Save Final Adapter ──────────────────────────────────────────────────
final_save = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_save)
processor.save_pretrained(final_save)
print(f"✅ Final adapter saved to {final_save}")

🚀 Starting Single-Stage SFT on VizWiz-LF synthetic data …
🔄 Auto-Resume: Found checkpoint at /kaggle/working/qwen3vl_qlora/checkpoint-600
📥 Resuming training from last saved state...
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


     [675/675 10:35:43, Epoch 3/3]


,
Step,Training Loss
600,1.184231
610,1.142294
620,1.037255
630,1.217009
640,1.097775
650,1.039370
660,1.039783
670,0.992799



✅ Hub push complete for epoch 3

✅ Training complete in 11.35h
   train_loss: 0.9827
✅ Final adapter saved to /kaggle/working/qwen3vl_qlora/checkpoint-675
